In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [2]:
path = "E:/Documents/dtms69/filtered_data.csv"

In [3]:
df = pd.read_csv(path, index_col=None)

In [4]:
df

,PEA_No,DATE_Time,KWH,SelfRead_KWH,VA,VB,VC,AA,AB,AC,KVARH_DEL,KVARH_REC,KWHExport,SelfRead_KWHEXP
0,6200022544,2025-12-01 00:00:00,0.007,12165.236,233.0,NaN,NaN,0.0,NaN,NaN,0.000,0.001,0.0,0.000
1,6200022544,2025-12-01 00:15:00,0.007,0.000,233.0,NaN,NaN,0.0,NaN,NaN,0.000,0.000,0.0,0.000
2,6200030933,2025-12-01 00:00:00,0.180,22210.018,234.0,NaN,NaN,2.0,NaN,NaN,0.045,0.000,0.0,0.000
3,6200030933,2025-12-01 00:15:00,0.110,0.000,234.0,NaN,NaN,2.0,NaN,NaN,0.045,0.000,0.0,0.000
4,6200031047,2025-12-01 00:00:00,0.074,33005.131,233.0,NaN,NaN,1.0,NaN,NaN,0.013,0.000,0.0,3569.829
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56420,6200032198,2025-12-31 23:45:00,0.017,0.000,233.0,NaN,NaN,0.0,NaN,NaN,0.000,0.011,0.0,0.000
56421,6200050504,2025-12-31 23:45:00,0.061,0.000,233.0,NaN,NaN,1.0,NaN,NaN,0.016,0.000,0.0,0.000
56422,6200061392,2025-12-31 23:45:00,0.800,0.000,232.0,231.0,234.0,14.0,0.0,1.0,0.214,0.000,0.0,0.000
56423,6200118626,2025-12-31 23:30:00,0.014,0.000,235.0,NaN,NaN,0.0,NaN,NaN,0.000,0.002,0.0,0.000


In [5]:
df2 = df.copy()

In [6]:
df2.insert(df2.columns.get_loc('KWH') + 1, 'kW average', df2['KWH'] * 4)

In [7]:
df2.insert(df2.columns.get_loc('KWHExport') + 1, 'kW exp average', df2['KWHExport'] * 4)

In [8]:
df2 = df2.drop(columns=['VA', 'VB', 'VC', 'SelfRead_KWH', 'SelfRead_KWHEXP','KWHExport', 'KVARH_REC', 'KVARH_DEL'])

In [10]:
finergy_list = [6200032193, 6200031047, 6200031051,
            6200031052, 6200031088, 6200032194, 6200031074]

df2['Finergy'] = np.where(df['PEA_No'].isin(finergy_list), 'true', 'false')

In [11]:
mapping = {
    'AA': 'A',
    'AB': 'B',
    'AC': 'C'
}

df2['Phase'] = (
    df2[['AA', 'AB', 'AC']]
    .notna()
    .idxmax(axis=1)
    .map(mapping)
)

In [12]:
df2[['AA', 'AB', 'AC']] = df2[['AA', 'AB', 'AC']].fillna(0)

In [13]:
df2['PEA_No'] = df2['PEA_No'].astype('string')

In [14]:
# print(df2['DATE_Time'].dtype)

df2['DATE_Time'] = pd.to_datetime(df2['DATE_Time'])

df2 = df2.set_index('DATE_Time')

In [15]:
df2

,PEA_No,KWH,kW average,AA,AB,AC,kW exp average,Finergy,Phase
DATE_Time,,,,,,,,,
2025-12-01 00:00:00,6200022544,0.007,0.028,0.0,0.0,0.0,0.0,false,A
2025-12-01 00:15:00,6200022544,0.007,0.028,0.0,0.0,0.0,0.0,false,A
2025-12-01 00:00:00,6200030933,0.180,0.720,2.0,0.0,0.0,0.0,false,A
2025-12-01 00:15:00,6200030933,0.110,0.440,2.0,0.0,0.0,0.0,false,A
2025-12-01 00:00:00,6200031047,0.074,0.296,1.0,0.0,0.0,0.0,true,A
...,...,...,...,...,...,...,...,...,...
2025-12-31 23:45:00,6200032198,0.017,0.068,0.0,0.0,0.0,0.0,false,A
2025-12-31 23:45:00,6200050504,0.061,0.244,1.0,0.0,0.0,0.0,false,A
2025-12-31 23:45:00,6200061392,0.800,3.200,14.0,0.0,1.0,0.0,false,A


In [16]:
# print(df2['ServicePointID'].nunique())
print(df2['PEA_No'].nunique())

19


In [17]:
df2.value_counts(['PEA_No'])

PEA_No    
6200022544    2976
6200030933    2976
6200031047    2976
6200031052    2976
6200031073    2976
6200031074    2976
6200031086    2976
6200031088    2976
6200031087    2976
6200032193    2976
6200031902    2976
6200032198    2976
6200050504    2976
6200032194    2976
6200032196    2976
6200061392    2976
6200118626    2976
6200031085    2952
6200031051    2881
Name: count, dtype: int64

In [18]:
df2 = df2.reset_index()

In [19]:
s = pd.to_datetime(df2['DATE_Time']).dropna().drop_duplicates().sort_values()

start = s.min().floor('15min')
end   = s.max().ceil('15min') - pd.Timedelta(minutes=15)  

full = pd.date_range(start, end, freq='15min')

In [20]:
day_count = pd.Series(s).groupby(pd.Series(s).dt.date).size()
incomplete_days = day_count[day_count < 96]
print(incomplete_days)

Series([], Name: DATE_Time, dtype: int64)


In [21]:
print(day_count.shape[0])

31


In [22]:
# ถ้า period ตรง 1 เดือนพอดี และคาดหวัง 96 จุด/วัน
expected = 96 * day_count.shape[0]         # คิดจากจำนวนวันจริงที่พบ
actual   = len(s)
print("expected:", expected, "actual:", actual, "missing:", expected - actual)

expected: 2976 actual: 2976 missing: 0


In [23]:
# 1 เดือน = 31 วัน → 2976 จุด
expected_fixed = 96 * 31
print("fixed_expected:", expected_fixed, "missing:", expected_fixed - actual)

fixed_expected: 2976 missing: 0


In [24]:
print(df2['AA'].isnull().sum())
print(df2['AB'].isnull().sum())
print(df2['AC'].isnull().sum())

0
0
0


In [25]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56425 entries, 0 to 56424
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   DATE_Time       56425 non-null  datetime64[ns]
 1   PEA_No          56425 non-null  string        
 2   KWH             56424 non-null  float64       
 3   kW average      56424 non-null  float64       
 4   AA              56425 non-null  float64       
 5   AB              56425 non-null  float64       
 6   AC              56425 non-null  float64       
 7   kW exp average  56424 non-null  float64       
 8   Finergy         56425 non-null  object        
 9   Phase           56425 non-null  object        
dtypes: datetime64[ns](1), float64(6), object(2), string(1)
memory usage: 4.3+ MB


In [26]:
df2.describe()

,DATE_Time,KWH,kW average,AA,AB,AC,kW exp average
count,56425,56424.000000,56424.000000,56425.000000,56425.0,56425.000000,56424.000000
mean,2025-12-16 11:58:19.512627200,0.137508,0.550033,1.927798,0.0,0.080266,0.108578
min,2025-12-01 00:00:00,0.000000,0.000000,-12.000000,0.0,0.000000,0.000000
25%,2025-12-08 17:30:00,0.018000,0.072000,0.000000,0.0,0.000000,0.000000
50%,2025-12-16 12:45:00,0.056000,0.224000,1.000000,0.0,0.000000,0.000000
75%,2025-12-24 06:15:00,0.132000,0.528000,2.000000,0.0,0.000000,0.000000
max,2025-12-31 23:45:00,2.148000,8.592000,38.000000,0.0,5.000000,2.812000
std,NaN,0.245903,0.983612,4.686560,0.0,0.366898,0.380733
